In [1]:
import pandas as pd
import numpy as np

from utils.config import load_config
from preprocessing.house_prices_preprocessing import HousePricesPreprocessor
from models.advanced_tuning import run_optuna_catboost

In [2]:
config = load_config()

data = pd.read_csv(config["paths"]["train"])

# --- разделяем target ---
target = "SalePrice"

# логарифмируем таргет
y = np.log1p(data[target])

X = data.drop(columns=[target])

# --- preprocessing ---
preprocessor = HousePricesPreprocessor(outlier_quantile=0.95)

X_prepared = preprocessor.fit_transform(X)

X_prepared.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageFinish_Unf,GarageFinish_Fin,GarageFinish_None,BsmtFinType1_GLQ,BsmtFinType1_ALQ,BsmtFinType1_Unf,BsmtFinType1_Rec,BsmtFinType1_BLQ,BsmtFinType1_None,BsmtFinType1_LwQ
0,0.073375,-0.208804,-0.332210,0.651479,-0.517200,1.050994,0.878668,0.739648,0.667140,-0.327561,...,0,0,0,1,0,0,0,0,0,0
1,-0.872563,0.646406,-0.009592,-0.071836,2.179628,0.156734,-0.429577,-0.654947,1.327216,-0.327561,...,0,0,0,0,1,0,0,0,0,0
2,0.073375,-0.037762,0.453294,0.651479,-0.517200,0.984752,0.830215,0.497729,0.133255,-0.327561,...,0,0,0,1,0,0,0,0,0,0
3,0.309859,-0.493874,-0.023619,0.651479,-0.517200,-1.863632,-0.720298,-0.654947,-0.521967,-0.327561,...,1,0,0,0,1,0,0,0,0,0
4,0.073375,0.874462,1.297710,1.374795,-0.517200,0.951632,0.733308,1.835402,0.543376,-0.327561,...,0,0,0,1,0,0,0,0,0,0


In [3]:
study, results_df = run_optuna_catboost(
    X=X_prepared,
    y=y,
    n_trials=500,
    n_splits=5,
)

Optuna tuning:   0%|          | 0/500 [00:00<?, ?it/s]

In [4]:
print("Лучшие параметры:", study.best_params)
print("Лучший RMSE:", study.best_value)

Лучшие параметры: {'iterations': 950, 'learning_rate': 0.05725960671224448, 'depth': 4, 'l2_leaf_reg': 9, 'random_strength': 3, 'subsample': 0.8}
Лучший RMSE: 0.12104847825302179


In [5]:
results_df.head()

,rank,number,mean_rmse,mean_best_rmse,mean_best_iteration,best_iteration_list,state,iterations,learning_rate,depth,l2_leaf_reg,random_strength,subsample
0,1,383,0.121048,0.121048,750.8,"[878, 920, 151, 918, 887]",TrialState.COMPLETE,950,0.057260,4,9,3,0.8
1,2,484,0.121183,0.121183,703.2,"[802, 837, 163, 928, 786]",TrialState.COMPLETE,930,0.056231,4,8,3,0.8
2,3,228,0.121186,0.121186,728.2,"[792, 1000, 159, 894, 796]",TrialState.COMPLETE,1000,0.058078,4,9,3,0.8
3,4,230,0.121223,0.121223,642.6,"[666, 847, 151, 779, 770]",TrialState.COMPLETE,1000,0.058528,4,9,3,0.8
4,5,244,0.121241,0.121241,711.8,"[802, 785, 160, 887, 925]",TrialState.COMPLETE,980,0.058637,4,9,3,0.8
